 ## Sudoku generator and solver - version 1 ##

In [128]:
sdk1 = '200709005600030002400000007002000400000104000000567000840000063029000810007000500'
sdk = '030000080000791000005000700004080500001204300200305008002508600000000000503010809'
def string_to_board(s):
    board = []
    for i in range(0,81,9):
        row = [int(x) for x in s[i:i+9]]
        board.append(row)
    return board


board = string_to_board(sdk)


In [129]:
def draw_board(board):
    for r in range(9):
        if r % 3 == 0 and r != 0:
            print("-"*21)

        row = ""
        for c in range(9):
            if c % 3 == 0 and c != 0:
                row += "| "

            val = board[r][c] if board[r][c] != 0 else "."
            row += str(val) + " "

        print(row)
draw_board(board)

. 3 . | . . . | . 8 . 
. . . | 7 9 1 | . . . 
. . 5 | . . . | 7 . . 
---------------------
. . 4 | . 8 . | 5 . . 
. . 1 | 2 . 4 | 3 . . 
2 . . | 3 . 5 | . . 8 
---------------------
. . 2 | 5 . 8 | 6 . . 
. . . | . . . | . . . 
5 . 3 | . 1 . | 8 . 9 


In [130]:
def get_candidates(board, row, col):
    if board[row][col] != 0:
        return set()

    digits = set(range(1, 10))

    row_nums = set(board[row])

    col_nums = {board[r][col] for r in range(9)}

    start_row = (row // 3) * 3
    start_col = (col // 3) * 3

    box_nums = {
        board[r][c]
        for r in range(start_row, start_row + 3)
        for c in range(start_col, start_col + 3)
    }

    used = row_nums | col_nums | box_nums
    used.discard(0)

    return digits - used

In [131]:
def get_all_candidates(board):
    candidates = [[set() for _ in range(9)] for _ in range(9)]

    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                candidates[row][col] = get_candidates(board, row, col)

    return candidates


In [132]:
# Naked Singles

def naked_singles(board, candidates):
    progress = False

    for row in range(9):
        for col in range(9):
            if board[row][col] == 0 and len(candidates[row][col]) == 1:
                value = next(iter(candidates[row][col]))
                board[row][col] = value
                print(f'Populated {value} at {row},{col}')
                progress = True

    return progress

In [133]:
# Hidden Singles

def hidden_singles_rows(board, candidates):

    for row in range(9):
        digit_positions = {digit: [] for digit in range(1, 10)}

        for col in range(9):
            if board[row][col] == 0:
                for digit in candidates[row][col]:
                    digit_positions[digit].append((row, col))

        for digit in range(1,10):
            if len(digit_positions[digit]) == 1:
                r, c = digit_positions[digit][0]
                board[r][c] = digit
                print(f'Hidden ROW single {digit} placed at position {r, c}')
                return True
    return False

def hidden_singles_columns(board, candidates):

    for col in range(9):
        digit_positions = {digit: [] for digit in range(1, 10)}

        for row in range(9):
            if board[row][col] == 0:
                for digit in candidates[row][col]:
                    digit_positions[digit].append((row, col))

        for digit in range(1,10):
            if len(digit_positions[digit]) == 1:
                r, c = digit_positions[digit][0]
                board[r][c] = digit
                print(f'Hidden COLUMN single {digit} placed at position {r, c}')
                return True
    return False

def hidden_singles_boxes(board, candidates):

    for box_row_start in range(0,9,3):
        for box_col_start in range(0,9,3):
            digit_positions = {digit: [] for digit in range(1, 10)}

            for row in range(box_row_start,box_row_start+3):
                for col in range(box_col_start,box_col_start+3):
                    if board[row][col] == 0:
                        for digit in candidates[row][col]:
                            digit_positions[digit].append((row, col))

            for digit in range(1,10):
                if len(digit_positions[digit]) == 1:
                    r, c = digit_positions[digit][0]
                    board[r][c] = digit
                    print(f'Hidden BOX single {digit} placed at position {r, c}')
                    return True
    return False

draw_board(board)


. 3 . | . . . | . 8 . 
. . . | 7 9 1 | . . . 
. . 5 | . . . | 7 . . 
---------------------
. . 4 | . 8 . | 5 . . 
. . 1 | 2 . 4 | 3 . . 
2 . . | 3 . 5 | . . 8 
---------------------
. . 2 | 5 . 8 | 6 . . 
. . . | . . . | . . . 
5 . 3 | . 1 . | 8 . 9 


In [134]:
# Naked PAIRS

def naked_pairs_rows(board, candidates):
    progress = False

    for row in range(9):
        pair_positions = {}

        for col in range(9):
            if len(candidates[row][col]) == 2 and board[row][col] == 0:
                pair = frozenset(candidates[row][col])

                if pair not in pair_positions:
                    pair_positions[pair] = []

                pair_positions[pair].append((row, col))

        for pair, cells in pair_positions.items():
            if len(cells) == 2:
                print(f'Found naked ROW pairs at {cells} in row {row}')
                pair_set = set(pair)
                pair_cells = set(cells)

                for col in range(9):
                    if (row,col) not in pair_cells and board[row][col] == 0:
                        before = candidates[row][col].copy()
                        candidates[row][col] -= pair_set

                        if candidates[row][col] != before:
                            print(f'Removed {pair_set} from {(row, col)}')
                            progress = True
    return progress

def naked_pairs_cols(board, candidates):
    progress = False

    for col in range(9):
        pair_positions = {}

        for row in range(9):
            if len(candidates[row][col]) == 2 and board[row][col] == 0:
                pair = frozenset(candidates[row][col])

                if pair not in pair_positions:
                    pair_positions[pair] = []

                pair_positions[pair].append((row, col))

        for pair, cells in pair_positions.items():
            if len(cells) == 2:
                print(f'Found naked COL pairs at {cells} in column {col}')
                pair_set = set(pair)
                pair_cells = set(cells)

                for row in range(9):
                    if (row,col) not in pair_cells and board[row][col] == 0:
                        before = candidates[row][col].copy()
                        candidates[row][col] -= pair_set

                        if candidates[row][col] != before:
                            print(f'Removed {pair_set} from {(row, col)}')
                            progress = True
    return progress

def naked_pairs_boxes(board, candidates):
    progress = False

    for box_row_start in range(0,9,3):
        for box_col_start in range(0,9,3):

            pair_positions = {}

            for row in range(box_row_start,box_row_start+3):
                for col in range(box_col_start,box_col_start+3):

                    if len(candidates[row][col]) == 2 and board[row][col] == 0:
                        pair = frozenset(candidates[row][col])

                        if pair not in pair_positions:
                            pair_positions[pair] = []

                        pair_positions[pair].append((row, col))

            for pair, cells in pair_positions.items():
                if len(cells) == 2:
                    pair_set = set(pair)
                    pair_cells = set(cells)
                    print(f'Found naked BOX pair {pair_set} at {cells} in box starting at {(box_row_start, box_col_start)}')

                    for row in range(box_row_start,box_row_start+3):
                        for col in range(box_col_start,box_col_start+3):
                            if (row,col) not in pair_cells and board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col] -= pair_set

                                if candidates[row][col] != before:
                                    print(f'Removed {pair_set} from {(row, col)}')
                                    progress = True
    return progress


In [135]:
# Hidden Pairs
def get_row_cells(row):
    return [(row, col) for col in range(9)]

def get_col_cells(col):
    return [(row, col) for row in range(9)]

def get_box_cells(start_row, start_col):
    return [
        (row, col)
        for row in range(start_row, start_row + 3)
        for col in range(start_col, start_col + 3)
    ]

def hidden_pairs_in_unit(board, candidates, unit_cells):
    progress = False
    digit_positions = {digit: [] for digit in range(1,10)}

    for row, col in unit_cells:
        if board[row][col] == 0:
            for digit in candidates[row][col]:
                digit_positions[digit].append((row, col))

    for d1 in range(1, 10):
        for d2 in range(d1 + 1, 10):
            pos1 = digit_positions[d1]
            pos2 = digit_positions[d2]

            if len(pos1) == 2 and pos1 == pos2:
                pair_cells = pos1
                hidden_pair = {d1,d2}

                for row, col in pair_cells:
                    before = candidates[row][col].copy()
                    candidates[row][col] &= hidden_pair

                    if candidates[row][col] != before:
                        print(f'Hidden pair {hidden_pair} reduced {(row, col)} from {before} to {candidates[row][col]}')
                        progress = True
    return progress

def hidden_pairs_rows(board, candidates):
    progress = False

    for row in range(9):
        if hidden_pairs_in_unit(board, candidates, get_row_cells(row)):
            progress = True

    return progress

def hidden_pairs_rows(board, candidates):
    progress = False

    for row in range(9):
        if hidden_pairs_in_unit(board, candidates, get_row_cells(row)):
            progress = True

    return progress

def hidden_pairs_cols(board, candidates):
    progress = False

    for col in range(9):
        if hidden_pairs_in_unit(board, candidates, get_col_cells(col)):
            progress = True

    return progress

def hidden_pairs_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if hidden_pairs_in_unit(board, candidates, get_box_cells(start_row, start_col)):
                progress = True

    return progress

def hidden_pairs(board, candidates):
    progress = False

    for row in range(9):
        if hidden_pairs_in_unit(board, candidates, get_row_cells(row)):
            progress = True

    for col in range(9):
        if hidden_pairs_in_unit(board, candidates, get_col_cells(col)):
            progress = True

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if hidden_pairs_in_unit(board, candidates, get_box_cells(start_row, start_col)):
                progress = True

    return progress

In [136]:
# Pointing Pairs/Triples

def pointing_pairs_triples(board, candidates):

    for r in range(0,9,3):
        for c in range(0,9,3):
            digit_positions = {digit: [] for digit in range(1,10)}

            for row in range(r, r+3):
                for col in range(c, c+3):
                    if board[row][col] == 0:
                        for digit in candidates[row][col]:
                            digit_positions[digit].append((row,col))

            for digit in range(1,10):
                cells = digit_positions[digit]

                # Rows
                if len(cells) == 2 and cells[0][0] == cells[1][0]:
                    target_row = cells[0][0]
                    print(f'Pointing pair of {digit}s in ROW {target_row}')
                    for x in range(9):
                        if not (c <= x < c+3) and board[target_row][x] == 0:
                            before = candidates[target_row][x].copy()
                            candidates[target_row][x].discard(digit)

                            if candidates[target_row][x] != before:
                                return True

                if len(cells) == 3 and cells[0][0] == cells[1][0] == cells[2][0]:
                    target_row = cells[0][0]
                    print(f'Pointing triple of {digit}s in ROW {target_row}')
                    for x in range(9):
                        if not (c <= x < c+3) and board[target_row][x] == 0:
                            before = candidates[target_row][x].copy()
                            candidates[target_row][x].discard(digit)

                            if candidates[target_row][x] != before:
                                return True

                # Columns
                if len(cells) == 2 and cells[0][1] == cells[1][1]:
                    target_col = cells[0][1]
                    print(f'Pointing pair of {digit}s in COL {target_col}')
                    for x in range(9):
                        if not (r <= x < r+3) and board[x][target_col] == 0:
                            before = candidates[x][target_col].copy()
                            candidates[x][target_col].discard(digit)

                            if candidates[x][target_col] != before:
                                return True

                if len(cells) == 3 and cells[0][1] == cells[1][1] == cells[2][1]:
                    target_col = cells[0][1]
                    print(f'Pointing triple of {digit}s in COL {target_col}')
                    for x in range(9):
                        if not (r <= x < r+3) and board[x][target_col] == 0:
                            before = candidates[x][target_col].copy()
                            candidates[x][target_col].discard(digit)

                            if candidates[x][target_col] != before:
                                return True

    return False

In [137]:
# Validation checks
def is_valid_row(board, row):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for cell in range(9):
        digits.add(board[row][cell])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_col(board, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()
    for cell in range(9):
        digits.add(board[cell][col])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_block(board, row, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for r in range(row, row+3):
        for c in range(col, col+3):
            digits.add(board[r][c])

    if digits == desired_set:
        return True
    else:
        return False

def is_solved(board):
    errors = 0
    for row in range(9):
        is_valid = is_valid_row(board, row)
        if not is_valid:
            errors += 1
            print(f'Board not valid at row {row+1}')

    for col in range(9):
        is_valid = is_valid_col(board, col)
        if not is_valid:
            errors += 1
            print(f'Board not valid at col {col+1}')

    for row in range(0,9,3):
        for col in range(0,9,3):
            is_valid = is_valid_block(board, row, col)
            if not is_valid:
                errors += 1
                print(f'Board not valid at block ({row+1}, {col+1})')

    if errors == 0:
        print(f'Board is solved, great success')
    else:
        print(f'Board is obviously not solved :(')


In [138]:
def run_solver(board):
    candidates = get_all_candidates(board)
    while True:

        if naked_singles(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_rows(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_columns(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_boxes(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if naked_pairs_rows(board, candidates):
            continue

        if naked_pairs_cols(board, candidates):
            continue

        if naked_pairs_boxes(board, candidates):
            continue

        if hidden_pairs(board, candidates):
            continue

        if pointing_pairs_triples(board, candidates):
            continue


        print("No more progress can be made.")
        draw_board(board)
        break

run_solver(board)


Hidden ROW single 3 placed at position (3, 0)
Hidden ROW single 5 placed at position (4, 1)
Hidden ROW single 8 placed at position (4, 0)
Hidden ROW single 9 placed at position (4, 7)
Hidden COLUMN single 1 placed at position (3, 3)
Hidden COLUMN single 8 placed at position (2, 3)
Hidden COLUMN single 9 placed at position (7, 3)
Hidden COLUMN single 5 placed at position (0, 4)
Hidden COLUMN single 9 placed at position (3, 5)
Hidden COLUMN single 9 placed at position (0, 6)
Hidden COLUMN single 9 placed at position (5, 2)
Found naked ROW pairs at [(4, 4), (4, 8)] in row 4
Found naked ROW pairs at [(5, 1), (5, 4)] in row 5
Removed {6, 7} from (5, 7)
Found naked ROW pairs at [(4, 4), (4, 8)] in row 4
Found naked ROW pairs at [(5, 1), (5, 4)] in row 5
Found naked ROW pairs at [(5, 6), (5, 7)] in row 5
Found naked COL pairs at [(3, 1), (5, 1)] in column 1
Removed {6, 7} from (1, 1)
Removed {6, 7} from (2, 1)
Removed {6, 7} from (6, 1)
Removed {6, 7} from (7, 1)
Removed {6, 7} from (8, 1)
Fo

Board is solved, great success
